# 03 Búsqueda vectorial con TF-IDF

## **Estudiante**: Eddy Alejandro Sangucho Coronel

## Importaciones

In [ ]:
# Importaciones y configuración
import os
import re
import math
import numpy as np
import pandas as pd
import nltk

from collections import Counter
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Versión de scikit-learn:", __import__("sklearn").__version__)

Versión de scikit-learn: 1.9.1


# 1.Recuperar y revisar el corpus

In [2]:
# 1.1 Carga del corpus y preprocesamiento

RUTA_CORPUS = r"C:\Users\Alejandro SC\Documents\txt-files.tar\txt-files\cache\epub"

def preprocesar(texto):
    """Minúsculas, conserva letras/números indicados y separa por espacios."""
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9áéíóúüñ]+", " ", texto)
    return texto.split()


textos_originales = {}
corpus_tokens = {}

# Buscar los primeros 18 documentos TXT
archivos_txt = []

if not os.path.isdir(RUTA_CORPUS):
    raise FileNotFoundError(
        "No se encontró RUTA_CORPUS. Verifica que la ruta exista y que "
        "la carpeta txt-files esté extraída."
    )

for raiz, carpetas, archivos in os.walk(RUTA_CORPUS):
    for nombre in archivos:
        if nombre.lower().endswith(".txt"):
            archivos_txt.append(os.path.join(raiz, nombre))

    if len(archivos_txt) >= 18:
        break

archivos_txt = sorted(archivos_txt)[:18]

print("Archivos encontrados:", len(archivos_txt))

if len(archivos_txt) != 18:
    raise ValueError(
        f"Se requieren 18 documentos TXT, pero se encontraron {len(archivos_txt)}."
    )

# Leer los documentos individualmente y conservar texto original
for ruta in archivos_txt:
    nombre = os.path.basename(ruta)

    with open(ruta, "r", encoding="utf-8", errors="replace") as archivo:
        texto = archivo.read()

    textos_originales[nombre] = texto
    corpus_tokens[nombre] = preprocesar(texto)

documentos = sorted(textos_originales.keys())
N = len(documentos)

# Tabla requerida: identificador, título y términos únicos
resumen_corpus = pd.DataFrame([
    {
        "Identificador de documento": doc,
        "Título": os.path.splitext(doc)[0].replace("-", " ").replace("_", " ").title(),
        "Número de términos únicos": len(set(corpus_tokens[doc]))
    }
    for doc in documentos
])

display(resumen_corpus)

# Estadísticas requeridas
longitudes = [len(corpus_tokens[doc]) for doc in documentos]
vocabulario_global = sorted({
    termino
    for doc in documentos
    for termino in corpus_tokens[doc]
})

print("\\nNúmero de documentos:", N)
print("Tamaño del vocabulario:", f"{len(vocabulario_global):,}")
print("Longitud mínima:", f"{min(longitudes):,}", "términos")
print("Longitud media:", f"{np.mean(longitudes):,.2f}", "términos")
print("Longitud máxima:", f"{max(longitudes):,}", "términos")
print("Total de tokens:", f"{sum(longitudes):,}")

for documento in documentos:
    print(documento, "->", len(corpus_tokens[documento]), "tokens")


Archivos encontrados: 18


,Identificador de documento,Título,Número de términos únicos
0,pg1.txt,Pg1,1212
1,pg10.txt,Pg10,13061
2,pg100.txt,Pg100,25070
3,pg1000.txt,Pg1000,13345
4,pg10000.txt,Pg10000,1870
5,pg10001.txt,Pg10001,2116
6,pg10002.txt,Pg10002,5339
7,pg10003.txt,Pg10003,6287
8,pg10004.txt,Pg10004,6868
9,pg10005.txt,Pg10005,7897


\nNúmero de documentos: 18
Tamaño del vocabulario: 54,861
Longitud mínima: 4,930 términos
Longitud media: 145,756.72 términos
Longitud máxima: 992,322 términos
Total de tokens: 2,623,621
pg1.txt -> 4930 tokens
pg10.txt -> 857542 tokens
pg100.txt -> 992322 tokens
pg1000.txt -> 105219 tokens
pg10000.txt -> 17741 tokens
pg10001.txt -> 8774 tokens
pg10002.txt -> 54750 tokens
pg10003.txt -> 66270 tokens
pg10004.txt -> 52360 tokens
pg10005.txt -> 74315 tokens
pg10006.txt -> 16491 tokens
pg10007.txt -> 31495 tokens
pg10008.txt -> 71312 tokens
pg10009.txt -> 91340 tokens
pg1001.txt -> 41465 tokens
pg10010.txt -> 13904 tokens
pg10011.txt -> 27586 tokens
pg10012.txt -> 95805 tokens


# 2. Implementación de funciones

In [3]:
from collections import Counter
import math

# 2.1 Implementación de funciones

def calcular_tf(tokens_documento):
    """Frecuencia absoluta de cada término en un documento."""
    return Counter(tokens_documento)


def calcular_df(corpus):
    """Número de documentos en los que aparece cada término."""
    frecuencias = Counter()

    for tokens in corpus.values():
        frecuencias.update(set(tokens))

    return frecuencias


def calcular_idf(df, numero_documentos):
    """IDF = ln(N / DF) + 1."""
    return {
        termino: math.log(numero_documentos / frecuencia) + 1
        for termino, frecuencia in df.items()
    }


def calcular_peso_tfidf(tf, idf):
    """TF-IDF = TF * IDF."""
    return {
        termino: frecuencia * idf[termino]
        for termino, frecuencia in tf.items()
        if termino in idf
    }


# Calcular TF, DF, IDF y TF-IDF
documento_ejemplo = documentos[0]

tf_ejemplo = calcular_tf(corpus_tokens[documento_ejemplo])
df_global = calcular_df(corpus_tokens)
idf_global = calcular_idf(df_global, N)

tfidf_ejemplo = calcular_peso_tfidf(
    tf_ejemplo,
    idf_global
)

print("Documento:", documento_ejemplo)
print("Términos TF:", len(tf_ejemplo))
print("Términos DF:", len(df_global))
print("Pesos TF-IDF:", len(tfidf_ejemplo))


# Mostrar los seis pesos TF-IDF más altos
seis_pesos = sorted(
    tfidf_ejemplo.items(),
    key=lambda x: x[1],
    reverse=True
)[:6]

tabla_seis_pesos = pd.DataFrame(
    seis_pesos,
    columns=["Término", "Peso TF-IDF"]
)

display(tabla_seis_pesos)


# Interpretación de uno de los términos
termino_ejemplo = seis_pesos[0][0]

print("\nInterpretación del término:", termino_ejemplo)
print("TF:", tf_ejemplo[termino_ejemplo])
print("DF:", df_global[termino_ejemplo])
print("IDF:", round(idf_global[termino_ejemplo], 6))
print("TF-IDF:", round(tfidf_ejemplo[termino_ejemplo], 6))

Documento: pg1.txt
Términos TF: 1212
Términos DF: 54861
Pesos TF-IDF: 1212


,Término,Peso TF-IDF
0,the,306.0
1,of,236.0
2,to,156.0
3,and,128.0
4,gutenberg,101.0
5,in,94.0



Interpretación del término: the
TF: 306
DF: 18
IDF: 1.0
TF-IDF: 306.0


# 3. Implementar TF-IDF para el corpus

In [4]:
# 3.1 Construcción del vocabulario y matriz TF-IDF

# Vocabulario ordenado alfabéticamente
vocabulario = sorted(df_global.keys())

indice_termino = {
    termino: indice
    for indice, termino in enumerate(vocabulario)
}

# Matriz TF
matriz_tf = np.zeros(
    (N, len(vocabulario)),
    dtype=np.float64
)

for fila, doc in enumerate(documentos):

    conteos = calcular_tf(corpus_tokens[doc])

    for termino, frecuencia in conteos.items():

        columna = indice_termino[termino]

        matriz_tf[fila, columna] = frecuencia


# Vector DF
df_vector = np.array([
    df_global[termino]
    for termino in vocabulario
])

# Vector IDF
idf_vector = np.log(N / df_vector) + 1

# Matriz TF-IDF
matriz_tfidf_manual = matriz_tf * idf_vector


# Comprobaciones
print("Matriz TF:", matriz_tf.shape)
print("Vector DF:", df_vector.shape)
print("Vector IDF:", idf_vector.shape)
print("Matriz TF-IDF:", matriz_tfidf_manual.shape)


# Verificar DF
df_desde_matriz = (matriz_tf > 0).sum(axis=0)

assert np.array_equal(
    df_vector,
    df_desde_matriz
)

assert np.all(matriz_tfidf_manual >= 0)


# Mostrar una pequeña parte de la matriz
columnas_muestra = min(10, len(vocabulario))

tabla_matriz = pd.DataFrame(
    matriz_tfidf_manual[:5, :columnas_muestra],
    index=documentos[:5],
    columns=vocabulario[:columnas_muestra]
)

display(tabla_matriz.round(4))

Matriz TF: (18, 54861)
Vector DF: (54861,)
Vector IDF: (54861,)
Matriz TF-IDF: (18, 54861)


,0,000,07042,1,10,100,1000,10000,10001,10002
pg1.txt,0.0000,2.0,1.0,53.0,1.5878,0.0000,0.0000,0.0000,0.0,0.0
pg10.txt,0.0000,1.0,1.0,2831.0,3364.5199,15.0245,0.0000,0.0000,0.0,0.0
pg100.txt,0.0000,1.0,1.0,79.0,3.1756,10.0163,2.5041,0.0000,0.0,0.0
pg1000.txt,0.0000,1.0,1.0,48.0,0.0000,0.0000,5.0082,0.0000,0.0,0.0
pg10000.txt,12.7889,1.0,1.0,54.0,4.7634,10.0163,0.0000,7.7807,0.0,0.0


# 4. Verificar con TfidfVectorizer

In [5]:
# 4.1 Comparación con TfidfVectorizer

documentos_preprocesados = [
    " ".join(corpus_tokens[doc])
    for doc in documentos
]

vectorizador = TfidfVectorizer(
    analyzer=str.split,
    lowercase=False,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    dtype=np.float64
)

# Ajustar y transformar
matriz_tfidf_sklearn = vectorizador.fit_transform(
    documentos_preprocesados
).toarray()

vocabulario_sklearn = vectorizador.get_feature_names_out()


# Alinear las columnas de ambas matrices
indices_sklearn = {
    termino: indice
    for indice, termino in enumerate(vocabulario_sklearn)
}

matriz_tfidf_sklearn_ordenada = matriz_tfidf_sklearn[
    :,
    [indices_sklearn[termino] for termino in vocabulario]
]


# Calcular máxima diferencia absoluta
diferencia_maxima = np.max(
    np.abs(
        matriz_tfidf_manual -
        matriz_tfidf_sklearn_ordenada
    )
)

print("Vocabularios idénticos:",
      vocabulario == list(vocabulario_sklearn))

print("Forma matriz manual:",
      matriz_tfidf_manual.shape)

print("Forma matriz scikit-learn:",
      matriz_tfidf_sklearn_ordenada.shape)

print("Máxima diferencia absoluta:",
      diferencia_maxima)


# Tolerancia
TOLERANCIA = 1e-10

print(
    "¿Coinciden las matrices?",
    diferencia_maxima <= TOLERANCIA
)

assert diferencia_maxima <= TOLERANCIA

Vocabularios idénticos: True
Forma matriz manual: (18, 54861)
Forma matriz scikit-learn: (18, 54861)
Máxima diferencia absoluta: 0.0
¿Coinciden las matrices? True


# 5.Representar consultas y calcular similitud coseno

In [6]:
# 5.1 Definición de cuatro consultas

consultas = {

    "Q1 - Aventura marítima": {
        "necesidad": "Encontrar contenido sobre ballenas, barcos, marineros y océano.",
        "texto": "whale whales ship ships sailor sailors sea ocean captain"
    },

    "Q2 - Relaciones y matrimonio": {
        "necesidad": "Encontrar historias sobre amor, relaciones y matrimonio.",
        "texto": "love marriage married husband wife affection courtship family"
    },

    "Q3 - Poder y ambición": {
        "necesidad": "Encontrar obras sobre reyes, poder, ambición, traición y asesinato.",
        "texto": "king power ambition throne murder betrayal crown prince"
    },

    "Q4 - Fantasía y Wonderland": {
        "necesidad": "Encontrar personajes y elementos de fantasía.",
        "texto": "rabbit queen hat wonderland alice cat tea"
    }
}


nombres_consultas = list(consultas.keys())

# Aplicar el mismo preprocesamiento
tokens_consultas = {
    nombre: preprocesar(datos["texto"])
    for nombre, datos in consultas.items()
}

textos_consultas = [
    " ".join(tokens_consultas[nombre])
    for nombre in nombres_consultas
]


# Transformar usando el vectorizador ya entrenado
matriz_consultas = vectorizador.transform(
    textos_consultas
).toarray()


print("Dimensiones de la matriz de consultas:",
      matriz_consultas.shape)

for nombre, datos in consultas.items():

    print("\n", nombre)
    print("Necesidad:", datos["necesidad"])
    print("Tokens:", tokens_consultas[nombre])

Dimensiones de la matriz de consultas: (4, 54861)

 Q1 - Aventura marítima
Necesidad: Encontrar contenido sobre ballenas, barcos, marineros y océano.
Tokens: ['whale', 'whales', 'ship', 'ships', 'sailor', 'sailors', 'sea', 'ocean', 'captain']

 Q2 - Relaciones y matrimonio
Necesidad: Encontrar historias sobre amor, relaciones y matrimonio.
Tokens: ['love', 'marriage', 'married', 'husband', 'wife', 'affection', 'courtship', 'family']

 Q3 - Poder y ambición
Necesidad: Encontrar obras sobre reyes, poder, ambición, traición y asesinato.
Tokens: ['king', 'power', 'ambition', 'throne', 'murder', 'betrayal', 'crown', 'prince']

 Q4 - Fantasía y Wonderland
Necesidad: Encontrar personajes y elementos de fantasía.
Tokens: ['rabbit', 'queen', 'hat', 'wonderland', 'alice', 'cat', 'tea']


In [7]:
# 5.2 Similitud coseno

def similitud_coseno_manual(vector_consulta, vector_documento):

    producto_punto = np.dot(
        vector_consulta,
        vector_documento
    )

    norma_consulta = np.linalg.norm(vector_consulta)
    norma_documento = np.linalg.norm(vector_documento)

    denominador = norma_consulta * norma_documento

    if denominador == 0:
        similitud = 0.0
    else:
        similitud = producto_punto / denominador

    return (
        float(similitud),
        float(producto_punto),
        float(norma_consulta),
        float(norma_documento)
    )


# Calcular similitud para las cuatro consultas
similitudes_manuales = np.zeros(
    (len(consultas), N),
    dtype=np.float64
)

for i, vector_consulta in enumerate(matriz_consultas):

    for j, vector_documento in enumerate(matriz_tfidf_manual):

        similitudes_manuales[i, j] = (
            similitud_coseno_manual(
                vector_consulta,
                vector_documento
            )[0]
        )


# Comparar con scikit-learn
similitudes_sklearn = cosine_similarity(
    matriz_consultas,
    matriz_tfidf_sklearn_ordenada
)

diferencia_coseno = np.max(
    np.abs(similitudes_manuales - similitudes_sklearn)
)

print("Máxima diferencia coseno:", diferencia_coseno)

assert diferencia_coseno <= TOLERANCIA


# Mostrar los términos compartidos y productos parciales
indice_q = 0

indice_d = int(
    np.argmax(similitudes_manuales[indice_q])
)

vector_q = matriz_consultas[indice_q]
vector_d = matriz_tfidf_manual[indice_d]


terminos_compartidos = [
    termino
    for k, termino in enumerate(vocabulario)
    if vector_q[k] != 0 and vector_d[k] != 0
]


productos_parciales = []

for termino in terminos_compartidos:

    indice = indice_termino[termino]

    peso_consulta = vector_q[indice]
    peso_documento = vector_d[indice]

    producto = peso_consulta * peso_documento

    productos_parciales.append({
        "Término compartido": termino,
        "Peso consulta": peso_consulta,
        "Peso documento": peso_documento,
        "Producto parcial": producto
    })


tabla_productos = pd.DataFrame(productos_parciales)

display(tabla_productos)


# Resultado final
sim, producto_punto, norma_q, norma_d = (
    similitud_coseno_manual(vector_q, vector_d)
)

print("\nConsulta:", nombres_consultas[indice_q])
print("Documento:", documentos[indice_d])
print("Producto punto:", producto_punto)
print("Norma consulta:", norma_q)
print("Norma documento:", norma_d)
print("Similitud coseno:", sim)

# Resumen conjunto solicitado para esta combinación consulta-documento
resumen_similitud = pd.DataFrame([{
    "Consulta": nombres_consultas[indice_q],
    "Documento": documentos[indice_d],
    "Producto punto": producto_punto,
    "Norma consulta": norma_q,
    "Norma documento": norma_d,
    "Similitud coseno": sim
}])

display(resumen_similitud)

Máxima diferencia coseno: 9.8879238130678e-16


,Término compartido,Peso consulta,Peso documento,Producto parcial
0,captain,2.098612,564.526706,1184.722682
1,ocean,1.492476,16.417241,24.502347
2,sailor,2.791759,25.125835,70.145288
3,sailors,2.280934,13.685603,31.215955
4,sea,1.182322,93.403403,110.432857
5,ship,1.810930,213.689766,386.977253
6,ships,2.280934,13.685603,31.215955



Consulta: Q1 - Aventura marítima
Documento: pg10008.txt
Producto punto: 1839.212337394752
Norma consulta: 6.715362486508522
Norma documento: 7224.828270888137
Similitud coseno: 0.037908350071845995


,Consulta,Documento,Producto punto,Norma consulta,Norma documento,Similitud coseno
0,Q1 - Aventura marítima,pg10008.txt,1839.212337,6.715362,7224.828271,0.037908


# 6.Construir el ranking usando TF-IDF

In [10]:
# 6.1 Recuperación de fragmentos y ranking top-5

def extraer_fragmento(texto_original, terminos_consulta, max_caracteres=280):

    texto_busqueda = texto_original.lower()

    posiciones = []

    for termino in terminos_consulta:

        patron = r"\b" + re.escape(termino) + r"\b"

        coincidencia = re.search(
            patron,
            texto_busqueda
        )

        if coincidencia:
            posiciones.append(coincidencia.start())

    if posiciones:

        inicio = max(0, min(posiciones) - 90)

        fragmento = texto_original[
            inicio:inicio + max_caracteres
        ]

    else:

        fragmento = texto_original[:max_caracteres]

    fragmento = re.sub(r"\s+", " ", fragmento).strip()

    if len(texto_original) > len(fragmento):
        fragmento += "..."

    return fragmento


# Construcción de rankings
rankings = {}

for i, nombre in enumerate(nombres_consultas):

    orden = np.argsort(
        -similitudes_manuales[i]
    )[:5]

    filas = []

    for posicion, indice_doc in enumerate(orden, start=1):

        doc = documentos[indice_doc]

        score_manual = float(
            similitudes_manuales[i, indice_doc]
        )

        score_sklearn = float(
            similitudes_sklearn[i, indice_doc]
        )

        filas.append({

            "Posición": posicion,

            "Identificador": doc,

            "Título": (
                doc.replace(".txt", "")
                   .replace("-", " ")
                   .title()
            ),

            "Puntuación": score_manual,

            "Puntuación scikit-learn": score_sklearn,

            "Fragmento": extraer_fragmento(
                textos_originales[doc],
                tokens_consultas[nombre]
            )
        })


    rankings[nombre] = pd.DataFrame(filas)

    print("\n" + "=" * 90)
    print(nombre)

    print(
        "Necesidad:",
        consultas[nombre]["necesidad"]
    )

    display(
        rankings[nombre][[
            "Posición",
            "Identificador",
            "Título",
            "Puntuación",
            "Puntuación scikit-learn",
            "Fragmento"
        ]].style.format({
            "Puntuación": "{:.8f}"
        })
    )


# Comprobación final
assert np.allclose(
    similitudes_manuales,
    similitudes_sklearn,
    atol=TOLERANCIA,
    rtol=0
)

print(
    "\nLas puntuaciones manuales y de scikit-learn coinciden."
)


Q1 - Aventura marítima
Necesidad: Encontrar contenido sobre ballenas, barcos, marineros y océano.


,Posición,Identificador,Título,Puntuación,Puntuación scikit-learn,Fragmento
0,1,pg10008.txt,Pg10008,0.03790835,0.037908,"SAMUEL HOPKINS ADAMS _Illustrations by Will Crawford_ 1907 CONTENTS PART ONE THE SEA RIDDLE I. DESERT SEAS II. THE ""LAUGHING LASS"" III. THE DEATH SHIP IV. THE SECOND PRIZE CREW V. THE DISAPPEARANCE VI. THE CASTAWAYS VII. THE FREE LANCE..."
1,2,pg100.txt,Pg100,0.00380329,0.003803,"air doth rehearse, Making a couplement of proud compare With sun and moon, with earth and sea’s rich gems: With April’s first-born flowers and all things rare, That heaven’s air in this huge rondure hems. O let me true in love but truly write, And then believe me, my love is as f..."
2,3,pg10005.txt,Pg10005,0.00347549,0.003475,onal character. CHAPTER IV. Continuation of the voyage--View of Europe; Atlantic Ocean; America--Speculations on the future destiny of the United States--Moral reflections-- Pacific Ocean--Hypothesis on the origin of the Moon. CHAPTER V. The voyage continued...
3,4,pg10.txt,Pg10,0.00232549,0.002325,"that may fly above the earth in the open firmament of heaven. 1:21 And God created great whales, and every living creature that moveth, which the waters brought forth abundantly, after their kind, and every winged fowl after his kind: and God saw that it was good. 1:22 And God..."
4,5,pg10012.txt,Pg10012,0.00225444,0.002254,"s the grand Central Valley of California, the waters of which have only one outlet to the sea through the Golden Gate. But with this general simplicity of features there is great complexity of hidden detail. The Coast Range, rising as a grand green barrier against the ocean, from..."



Q2 - Relaciones y matrimonio
Necesidad: Encontrar historias sobre amor, relaciones y matrimonio.


,Posición,Identificador,Título,Puntuación,Puntuación scikit-learn,Fragmento
0,1,pg100.txt,Pg100,0.01821358,0.018214,AND DEATH OF KING JOHN THE TRAGEDY OF JULIUS CAESAR THE TRAGEDY OF KING LEAR LOVE’S LABOUR’S LOST THE TRAGEDY OF MACBETH MEASURE FOR MEASURE THE MERCHANT OF VENICE THE MERRY WIVES OF WINDSOR A MIDSUMMER NIGHT’S DREAM MUCH ADO ABOUT NOTHING...
1,2,pg10006.txt,Pg10006,0.01617323,0.016173,"NI BOCCACCIO TRANSLATED BY JAMES C. BROGAN 1907. INTRODUCTION Youth, beauty, and love, wit, gayety and laughter, are the component parts of the delightful picture conjured up by the mere name of Giovanni Boccaccio, the prince of story-tellers for all generations of men...."
2,3,pg10003.txt,Pg10003,0.00940827,0.009408,II. M. WADDINGTON AS MINISTER OF PUBLIC INSTRUCTION IV. THE SOCIAL SIDE OF A MINISTER'S WIFE V. A REPUBLICAN VICTORY AND A NEW MINISTRY VI. THE EXPOSITION YEAR VII. THE BERLIN CONGRESS VIII. GAIETIES AT THE QUAI D'ORSAY IX. M. WADDINGTON AS PRIME MINISTER X. PARLIAME...
3,4,pg10000.txt,Pg10000,0.00904151,0.009042,"on demands and the revenues from the land can reasonably bear. (6) Heirs may be given in marriage, but not to someone of lower social standing. Before a marriage takes place, it shall be’ made known to the heir’s next-of-kin. (7) At her husband’s death, a widow may have her mar..."
4,5,pg10004.txt,Pg10004,0.00672909,0.006729,"s to be Victor in the great combats of the race. It is the spirit of courage, energy, and love. Nothing is too hard for it, nothing too distasteful, nothing too insignificant. Through all the course of duty it spurs one to do one's best. Its essence is to overcome. This is the in..."



Q3 - Poder y ambición
Necesidad: Encontrar obras sobre reyes, poder, ambición, traición y asesinato.


,Posición,Identificador,Título,Puntuación,Puntuación scikit-learn,Fragmento
0,1,pg100.txt,Pg100,0.02422966,0.024230,"E COMEDY OF ERRORS THE TRAGEDY OF CORIOLANUS CYMBELINE THE TRAGEDY OF HAMLET, PRINCE OF DENMARK THE FIRST PART OF KING HENRY THE FOURTH THE SECOND PART OF KING HENRY THE FOURTH THE LIFE OF KING HENRY THE FIFTH THE FIRST PART OF HENRY THE SIXTH THE..."
1,2,pg10.txt,Pg10,0.01101727,0.011017,"The Project Gutenberg eBook of The King James Version of the Bible This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the..."
2,3,pg10003.txt,Pg10003,0.00910040,0.009100,"before using this eBook. Title: My First Years as a Frenchwoman, 1876-1879 Author: Mary King Waddington Release date: November 1, 2003 [eBook #10003] Most recently updated: October 28, 2024 Language: English Other information and formats: www.gutenb..."
3,4,pg10000.txt,Pg10000,0.00709315,0.007093,"ntain a dozen different versions. The Text of Magna Carta JOHN, by the grace of God King of England, Lord of Ireland, Duke of Normandy and Aquitaine, and Count of Anjou, to his archbishops, bishops, abbots, earls, barons, justices, foresters, sheriffs, stewards, servants, a..."
4,5,pg10004.txt,Pg10004,0.00671901,0.006719,"GHER CONQUEST [CUTLER] _The Son of God goes forth to war, A kingly crown to gain: His blood-red banner streams afar: Who follows in His train? Who best can drink his cup of woe, Triumphant over pain; Who patient bears h..."



Q4 - Fantasía y Wonderland
Necesidad: Encontrar personajes y elementos de fantasía.


,Posición,Identificador,Título,Puntuación,Puntuación scikit-learn,Fragmento
0,1,pg100.txt,Pg100,0.00625728,0.006257,"e and less: Thou mak’st faults graces, that to thee resort: As on the finger of a throned queen, The basest jewel will be well esteemed: So are those errors that in thee are seen, To truths translated, and for true things deemed. How many lambs might the stern wolf betray, If lik..."
1,2,pg10003.txt,Pg10003,0.00344421,0.003444,"AMBER OF DEPUTIES MEETING AS THE NATIONAL ASSEMBLY THE ELYSÉE PALACE, PARIS HER MAJESTY QUEEN VICTORIA, ABOUT 1879 M. DE FREYCINET MME. SADI CARNOT PRESIDENT SADI CARNOT MY FIRST YEARS AS A FRENCHWOMAN I WHEN MACMAHON WAS PRESIDENT I was married in Paris in Novemb..."
2,3,pg10011.txt,Pg10011,0.00223264,0.002233,the oven with 1 cup of beef broth. Baste often and serve very hot. 14.--Spanish Stewed Rabbit. Clean and parboil 2 rabbits; then cut into pieces. Sprinkle with flour and fry in hot lard. Remove the rabbits. Add chopped tomato and onion to the sauce; mix with flour; let fry; a...
3,4,pg10009.txt,Pg10009,0.00206009,0.002060,"vin',' and away he bolted like a steam engine, down the lake. When he straightened up, my hat flew more than sixty yards behind me, and the way I came down into the bottom of the boat was anything but pleasant. Away we tore down towards the outlet, the boat cuttin' and plowin' th..."
4,5,pg10007.txt,Pg10007,0.00179196,0.001792,"o appeared to be lifeless. My dear old father was already beside the elder lady, with his hat in his hand, evidently tendering his aid and the resources of his schloss. The lady did not appear to hear him, or to have eyes for anything but the slender girl who was being placed aga..."



Las puntuaciones manuales y de scikit-learn coinciden.


# Conclución

En conclusión, los resultados obtenidos demuestran que la implementación del modelo TF-IDF permitió representar y analizar correctamente un corpus de 18 documentos, compuesto por 2.623.621 términos y un vocabulario de 54.861 palabras únicas. Se comprobó que los cálculos realizados manualmente coinciden exactamente con los obtenidos mediante la biblioteca Scikit-learn, con una diferencia máxima de 0.0 en la matriz TF-IDF y una diferencia prácticamente nula en la similitud coseno (9.88 × 10⁻¹⁶). Además, mediante cuatro consultas temáticas se logró calcular la similitud entre las búsquedas y los documentos, generando rankings con los cinco resultados más relevantes para cada consulta. Esto demuestra que el modelo implementado funciona correctamente y permite recuperar información de manera organizada según la relación entre los términos de búsqueda y el contenido de los documentos.